In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

fonts = sorted({f.name for f in fm.fontManager.ttflist})
korean_candidates = ['Malgun Gothic', 'NanumGothic', 'NanumBarunGothic', 'AppleGothic']
found = next((c for c in korean_candidates if c in fonts), None)
if found is None:
    raise RuntimeError('한글 폰트를 찾지 못했습니다.')
plt.rcParams['font.family'] = found
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(font=found, rc={'axes.unicode_minus': False})
print('폰트:', found)

## 1. 데이터 로드

In [ ]:
df   = pd.read_csv(r'data/Membership_v2.csv')
vh   = pd.read_csv(r'data/View_History.csv')
mm   = pd.read_csv(r'data/Movie_Master_kobis.csv')
um   = pd.read_csv(r'data/User_Mapping.csv')

print('Membership_v2:', df.shape)
print('View_History :', vh.shape)
print('Movie_Master :', mm.shape)
print('User_Mapping :', um.shape)

## 2. View_History 집계 — avg_rewatch_ratio / avg_release_year / new_movie_ratio

In [ ]:
# RELEASE_MONTH: YYYYMM 형식 → 연도 추출
mm['release_year'] = (mm['RELEASE_MONTH'].astype(str).str[:4]).astype(float)

# View_History에 release_year 조인
vh_mm = vh.merge(mm[['MOVIE_ID', 'release_year']], on='MOVIE_ID', how='left')

# 신작 기준: 2019년 이후 개봉
NEW_YEAR = 2019
vh_mm['is_new_movie'] = (vh_mm['release_year'] >= NEW_YEAR).astype(float)

# USER_ID 기준 집계
vh_agg = vh_mm.groupby('USER_ID').agg(
    avg_rewatch_ratio=('WATCH_SEQ', lambda x: (x > 1).sum() / len(x)),
    avg_release_year =('release_year', 'mean'),
    new_movie_ratio  =('is_new_movie', 'mean'),
).reset_index()

# USER_ID → user_no 매핑 (User_Mapping: uid=user_no, USER_ID=숫자)
vh_agg = vh_agg.merge(um[['USER_ID', 'uid']], on='USER_ID', how='left')

print(vh_agg.shape)
vh_agg.head(3)

## 3. Membership_v2에 신규 피처 추가

In [ ]:
df = df.merge(
    vh_agg[['uid', 'avg_rewatch_ratio', 'avg_release_year', 'new_movie_ratio']],
    left_on='user_no', right_on='uid', how='left'
)

# 시청 이력 없는 유저 fillna
df['avg_rewatch_ratio'] = df['avg_rewatch_ratio'].fillna(0)
df['avg_release_year']  = df['avg_release_year'].fillna(df['avg_release_year'].median())
df['new_movie_ratio']   = df['new_movie_ratio'].fillna(0)

# cold_start: 시청 이력 없음
df['cold_start'] = (df['has_watch_history'] == 0).astype(int)

# active_days: watch_days_count와 동일 개념
df['active_days'] = df['watch_days_count']

# watch_per_day: 하루 평균 시청 횟수
df['watch_per_day'] = (
    df['total_watch_count'] / df['duration_days'].replace(0, np.nan)
).fillna(0)

# 플랜 원-핫
df['is_basic']    = (df['plan_tier'] == 'basic').astype(int)
df['is_standard'] = (df['plan_tier'] == 'standard').astype(int)
df['is_premium']  = (df['plan_tier'] == 'premium').astype(int)

# plan × promotion_yn 상호작용
df['plan_x_promo'] = df[['is_basic','is_standard','is_premium']].idxmax(axis=1).map(
    {'is_basic': 1, 'is_standard': 2, 'is_premium': 3}
) * df['promotion_yn']

# currency_type 인코딩
df['is_usd'] = (df['currency_type'] == 'USD').astype(int)

# age_group (없으면 생성)
if 'age_group' not in df.columns:
    df['age_group'] = (df['age'] // 10) * 10

print('피처 생성 완료')
print(df.shape)

## 4. 최종 피처 선택

In [ ]:
feature_cols = [
    # 원본
    'billing_method', 'concurrent_streams', 'promotion_yn', 'is_churn_prevented',
    'is_user_verified', 'gender', 'age', 'reg_hour', 'duration_days',

    # 전처리 파생 (Membership)
    'is_usd', 'is_promotional_price', 'is_night_signup',
    'reg_weekday', 'is_same_day_cancel', 'age_group',

    # 전처리 파생 (View_History 조인)
    'total_watch_count', 'total_watch_duration', 'unique_movies',
    'avg_duration', 'watch_days_count', 'has_watch_history',

    # 신규 파생 — 온보딩/시간
    'signup_to_first_watch', 'recency', 'cold_start',
    'dur_w1', 'dur_w2', 'dur_w3', 'retention_w2', 'retention_w3',

    # 신규 파생 — 시청 행동
    'active_days', 'avg_rewatch_ratio', 'watch_per_day', 'avg_release_year',
    'watch_density', 'weekday_watch_ratio', 'binge_score', 'completion_rate',

    # 신규 파생 — 상호작용
    'stream_watch_interaction', 'plan_x_promo',

    # 신규 파생 — 신작
    'new_movie_ratio',

    # 장르 원-핫
    'genre_SF', 'genre_다큐', 'genre_드라마', 'genre_로맨스', 'genre_스릴러',
    'genre_애니메이션', 'genre_액션', 'genre_어드벤처', 'genre_코미디', 'genre_판타지',

    # 플랜 원-핫
    'is_basic', 'is_standard', 'is_premium',
]

# 실제 존재하는 컬럼만
missing = [c for c in feature_cols if c not in df.columns]
if missing:
    print('⚠ 없는 컬럼:', missing)
feature_cols = [c for c in feature_cols if c in df.columns]

df_feat = df[feature_cols]
print(f'사용 피처 수: {len(feature_cols)}개')
print(df_feat.shape)

## 5. 상관계수 행렬 히트맵

In [ ]:
corr = df_feat.corr()

fig, ax = plt.subplots(figsize=(26, 22))
sns.heatmap(
    corr,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.4,
    annot_kws={'size': 7},
    ax=ax
)
ax.set_title(f'수치형 피처 간 상관계수 행렬 ({len(feature_cols)}개 피처)', fontsize=14, pad=15)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

## 6. |상관계수| >= 0.7 쌍 출력

In [ ]:
high_corr = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)
high_corr.columns = ['col1', 'col2', 'corr']
high_corr = high_corr[high_corr['corr'].abs() >= 0.7].sort_values('corr', ascending=False)

print(f'[ |상관계수| >= 0.7 인 쌍: {len(high_corr)}개 ]\n')
print(high_corr.to_string(index=False) if not high_corr.empty else '해당 없음')